In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [5]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../../data/nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

107


In [6]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑  ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
F

{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': '../../data/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

516

In [22]:
from langchain_ollama import OllamaEmbeddings


embeddings = OllamaEmbeddings(model="llama3")

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 4096

[-0.0043087127, -0.029127879, -0.01385499, 0.0040667765, -0.005861809, -0.02568284, -0.0069189444, -0.006874306, -0.034442145, -0.00085926877]


In [25]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

ids = vector_store.add_documents(documents=all_splits)



In [26]:
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

page_content='Table of Contents
INTERNATIONAL MARKETS
For fiscal 2023, non-U.S. NIKE Brand and Converse sales accounted for approximately 57% of total revenues, compared to 60% and 61% for fiscal 2022 and fiscal 2021,
respectively. We sell our products to retail accounts through our own NIKE Direct operations and through a mix of independent distributors, licensees and sales
representatives around the world. We sell to thousands of retail accounts and ship products from 67 distribution centers outside of the United States. Refer to Item 2.
Properties for further information on distribution facilities outside of the United States. During fiscal 2023, NIKE's three largest customers outside of the United States
accounted for approximately 14% of total non-U.S. sales.
In addition to NIKE-owned and Converse-owned digital commerce platforms in over 40 countries, our NIKE Direct and Converse direct to consumer businesses operate
the following number of retail stores outside the United States:

In [28]:
results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

page_content='Table of Contents
INTERNATIONAL MARKETS
For fiscal 2023, non-U.S. NIKE Brand and Converse sales accounted for approximately 57% of total revenues, compared to 60% and 61% for fiscal 2022 and fiscal 2021,
respectively. We sell our products to retail accounts through our own NIKE Direct operations and through a mix of independent distributors, licensees and sales
representatives around the world. We sell to thousands of retail accounts and ship products from 67 distribution centers outside of the United States. Refer to Item 2.
Properties for further information on distribution facilities outside of the United States. During fiscal 2023, NIKE's three largest customers outside of the United States
accounted for approximately 14% of total non-U.S. sales.
In addition to NIKE-owned and Converse-owned digital commerce platforms in over 40 countries, our NIKE Direct and Converse direct to consumer businesses operate
the following number of retail stores outside the United States:

In [51]:

# results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
results = vector_store.similarity_search_with_score("For fiscal 2023, non-U.S. NIKE Brand and Converse sales accounted for approximately 57% of total revenues,")

doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.8502313494682312

page_content='Table of Contents
Critical Audit Matters
The critical audit matter communicated below is a matter arising from the current period audit of the consolidated financial statements that was communicated or required
to be communicated to the audit committee and that (i) relates to accounts or disclosures that are material to the consolidated financial statements and (ii) involved our
especially challenging, subjective, or complex judgments. The communication of critical audit matters does not alter in any way our opinion on the consolidated financial
statements, taken as a whole, and we are not, by communicating the critical audit matter below, providing a separate opinion on the critical audit matter or on the accounts
or disclosures to which it relates.
Accounting for Income Taxes
As described in Notes 1 and 7 to the consolidated financial statements, the Company recorded income tax expense of $1,131 million for the year ended May 31, 2023,' metada

In [50]:
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")
embedding = embeddings.embed_query("For fiscal 2023, non-U.S. NIKE Brand and Converse sales accounted for approximately of total revenues,")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='Table of Contents
Critical Audit Matters
The critical audit matter communicated below is a matter arising from the current period audit of the consolidated financial statements that was communicated or required
to be communicated to the audit committee and that (i) relates to accounts or disclosures that are material to the consolidated financial statements and (ii) involved our
especially challenging, subjective, or complex judgments. The communication of critical audit matters does not alter in any way our opinion on the consolidated financial
statements, taken as a whole, and we are not, by communicating the critical audit matter below, providing a separate opinion on the critical audit matter or on the accounts
or disclosures to which it relates.
Accounting for Income Taxes
As described in Notes 1 and 7 to the consolidated financial statements, the Company recorded income tax expense of $1,131 million for the year ended May 31, 2023,' metadata={'producer': 'EDGRpdf Se

In [ ]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

[[Document(id='7f850d93-f2e0-47bc-aed3-9386489118ff', metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': '../../data/nke-10k-2023.pdf', 'total_pages': 107, 'page': 5, 'page_label': '6', 'start_index': 0}, page_content="Table of Contents\nINTERNATIONAL MARKETS\nFor fiscal 2023, non-U.S. NIKE Brand and Converse sales accounted for approximately 57% of total revenues, compared to 60% and 61% for fiscal 2022 and fiscal 2021,\nrespectively. We sell our products to retail accounts through our own NIKE Direct operations and through a mix of independent distributors, licensees and sales\nrepresentatives around the world. We sell

In [45]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
        "How many were the sales of Nike and Converse in Fiscal year 2023 for Non-US out of total?"
    ],
)

[[Document(id='7f850d93-f2e0-47bc-aed3-9386489118ff', metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': '../../data/nke-10k-2023.pdf', 'total_pages': 107, 'page': 5, 'page_label': '6', 'start_index': 0}, page_content="Table of Contents\nINTERNATIONAL MARKETS\nFor fiscal 2023, non-U.S. NIKE Brand and Converse sales accounted for approximately 57% of total revenues, compared to 60% and 61% for fiscal 2022 and fiscal 2021,\nrespectively. We sell our products to retail accounts through our own NIKE Direct operations and through a mix of independent distributors, licensees and sales\nrepresentatives around the world. We sell

In [49]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import chain
from langchain_ollama.llms import OllamaLLM

# Initialize the Ollama model
llm = OllamaLLM(model="gemma3:12b")

@chain
def retriever(query: str) -> List[Document]:
    """Retrieve the most relevant documents for the query."""
    return vector_store.similarity_search(query, k=1)

def generate_response(query: str) -> str:
    """Generate a response using Ollama based on the query and retrieved context."""
    # Retrieve relevant documents
    retrieved_docs = retriever.batch(query)
    
    # Combine the content of the retrieved documents
    context = "\n\n".join(doc[0].page_content for doc in retrieved_docs)
    
    # Format the input for the Ollama model
    prompt = f"Context:\n{context}\n\nQuestion:\n{query}\n\nAnswer:"
    
    # Generate a response using the Ollama model
    response = llm.invoke(prompt)
    
    return response

# Example usage
# query = "How many distribution centers does Nike have in the US?"
query = "Percentage of Nike and Converse in Fiscal year 2023 for Non-US out of total revenue?"
response = generate_response(query)
print(response)

Okay, let's break down how to estimate the percentage of Nike and Converse revenue from outside the U.S. for fiscal year 2023 based on the provided text.

Here's the challenge: the provided text doesn't give us those exact figures. However, it *does* provide some clues that we can use to make a reasonable estimate.

**Key Information & Approach**

1.  **Geographic Revenue Mix:** We need to determine the proportion of Nike's revenue that comes from outside the U.S.
2.  **Nike vs. Converse:**  We need to know what percentage of Nike's revenue is generated by Converse.
3.  **Estimated Calculation:** Multiply the geographic proportion by the proportion of Converse revenue.

**Let's find the information:**

The provided text doesn't explicitly state the information, so it's not possible to provide an accurate answer based solely on the provided text.



**Disclaimer:** This estimation is based on publicly available information. Actual figures may vary.
